In [3]:
"""
Build a stakeholder-facing Excel workbook with granular NAICS/SOC detail -
the "how many roles were coded as Human Resources Specialist" question
the sector-level charts can't answer on their own.

Includes Role Description on Raw Detail so any stakeholder can read the
actual text a coding was based on and judge its trustworthiness for
themselves, rather than taking the NAICS/SOC match on faith.

Combines BOTH data sources now that Cultural Corps coding (Step 3) is
done: the 4 core hubs (AllRoles_nioccs_ready/coded) AND Cultural Corps
(cc_step2_ready_for_nioccs / cc_step3_coded). This is a lighter-weight
combine than the full Step 4 merge into the citywide-comparison pipeline
(that's still a separate step) - this just needs the raw coded rows
together for one workbook.

Sheets:
  - Read Me       : what this is, data sources, caveats
  - Raw Detail    : one row per coded placement, including Role
                    Description - the underlying data
  - SOC Summary   : count of placements per SOC (occupation) title,
                    broken out by hub, via COUNTIFS formulas (not
                    hardcoded numbers) so it recalculates if Raw Detail
                    changes
  - NAICS Summary : same idea, by NAICS (industry) title
"""

import re
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# XLSX/XML doesn't permit raw control characters (this is a format
# restriction, not an openpyxl quirk - xlsxwriter hits the same limit,
# it just doesn't always raise on it). Free-text descriptions pasted from
# Word docs or web forms occasionally carry invisible control characters
# (vertical tabs, form feeds, etc.) that trip this. Strip only those -
# normal punctuation, em-dashes, bullets, and Unicode text are untouched.
ILLEGAL_XLSX_CHARACTERS_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")


def sanitize_for_excel(value):
    if isinstance(value, str):
        return ILLEGAL_XLSX_CHARACTERS_RE.sub("", value)
    return value


# Core hub sources
READY_PATH = "../../data/AllRoles_nioccs_ready.csv"
CODED_PATH = "../../data/AllRoles_nioccs_coded_with_education.csv"

# Cultural Corps sources (Step 2 / Step 3 outputs)
CC_READY_PATH = "../../data/cultural_corps/cc_step2_ready_for_nioccs.csv"
CC_CODED_PATH = "../../data/cultural_corps/cc_step3_coded.csv"

OUT_PATH = "../../data/Sector_Detail_Workbook.xlsx"

FONT_NAME = "Arial"
HEADER_FILL = PatternFill(start_color="1E2761", end_color="1E2761", fill_type="solid")
HEADER_FONT = Font(name=FONT_NAME, bold=True, color="FFFFFF")
BODY_FONT = Font(name=FONT_NAME)
WRAP_FONT_ALIGN = Alignment(wrap_text=True, vertical="top")
TITLE_FONT = Font(name=FONT_NAME, bold=True, size=14, color="1E2761")


def style_header_row(ws, row_num, n_cols):
    for col in range(1, n_cols + 1):
        cell = ws.cell(row=row_num, column=col)
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center")


def autofit_columns(ws, widths):
    for i, w in enumerate(widths, start=1):
        ws.column_dimensions[get_column_letter(i)].width = w


# --- Load and merge core hub data ---
ready = pd.read_csv(READY_PATH, encoding="utf-8-sig")
coded = pd.read_csv(CODED_PATH, encoding="utf-8-sig")
core = coded.merge(
    ready[["Opportunity Identifier", "Cohort Year", "Role Description"]],
    on="Opportunity Identifier", how="left",
)
core = core[core["API Call Error"].isna()]
print(f"Core hub rows (successfully coded): {len(core)}")

# --- Load and merge Cultural Corps data ---
cc_ready = pd.read_csv(CC_READY_PATH, encoding="utf-8-sig")
cc_coded = pd.read_csv(CC_CODED_PATH, encoding="utf-8-sig")
cc = cc_coded.merge(
    cc_ready[["Opportunity Identifier", "Cohort Year", "Role Description"]],
    on="Opportunity Identifier", how="left",
)
cc = cc[cc["API Call Error"].isna()]
# Cultural Corps hasn't been through the education-tagging step yet -
# add the column as blank so it lines up with the core hub columns
# rather than silently dropping the column for these rows.
if "Typical Education Needed For Entry" not in cc.columns:
    cc["Typical Education Needed For Entry"] = None
print(f"Cultural Corps rows (successfully coded): {len(cc)}")

merged = pd.concat([core, cc], ignore_index=True)
print(f"Combined total: {len(merged)}")

hubs = sorted(merged["Program Hub Category"].dropna().unique().tolist())
print(f"Hubs present in this data: {hubs}")

wb = Workbook()

# =====================================================================
# Sheet: Read Me
# =====================================================================
ws = wb.active
ws.title = "Read Me"
ws["A1"] = "Sector Detail Workbook"
ws["A1"].font = TITLE_FONT
ws["A3"] = "Purpose"
ws["A3"].font = Font(name=FONT_NAME, bold=True)
ws["A4"] = ("The sector-level charts and slides show broad hub/industry totals. This workbook goes "
            "one level deeper - every placement's specific coded occupation (SOC) and industry (NAICS) "
            "title - so a question like \"how many roles were coded as Human Resources Specialist\" "
            "has a direct answer.")
ws["A4"].alignment = Alignment(wrap_text=True)
ws.row_dimensions[4].height = 40

ws["A6"] = "How to use this"
ws["A6"].font = Font(name=FONT_NAME, bold=True)
ws["A7"] = ("Raw Detail is the underlying data, including each posting's actual Role Description - "
            "read it directly to judge whether a NAICS/SOC match looks right, rather than taking it "
            "on faith. SOC Summary and NAICS Summary are pivot-style counts built with COUNTIFS "
            "formulas referencing Raw Detail directly - if you filter or add rows to Raw Detail, the "
            "summary counts update automatically.")
ws["A7"].alignment = Alignment(wrap_text=True)
ws.row_dimensions[7].height = 55

ws["A9"] = "Coverage"
ws["A9"].font = Font(name=FONT_NAME, bold=True)
ws["A10"] = ("Includes both the 4 core hubs (Healthcare, STEM and Green, Community and Social "
             "Services, Marketing and Communications) and Cultural Corps. Cultural Corps rows have "
             "not yet been through the education-level tagging step, so \"Typical Education Needed "
             "For Entry\" will show blank for those rows until that's run separately.")
ws["A10"].alignment = Alignment(wrap_text=True)
ws.row_dimensions[10].height = 55

ws["A12"] = "Data source"
ws["A12"].font = Font(name=FONT_NAME, bold=True)
ws["A13"] = "NIOCCS (CDC/NIOSH) auto-coding of internship postings against NAICS 2017 and SOC 2018."
ws["A14"] = f"Generated from {len(merged)} successfully coded placements across {len(hubs)} hubs."

for row in ws.iter_rows():
    for cell in row:
        if cell.font.name != FONT_NAME:
            cell.font = Font(name=FONT_NAME)
autofit_columns(ws, [90])

# =====================================================================
# Sheet: Raw Detail
# =====================================================================
ws = wb.create_sheet("Raw Detail")
detail_cols = [
    "Opportunity Identifier", "Role Name", "Program Hub Category", "Cohort Year",
    "NAICS Code", "NAICS Title", "NAICS Match Probability",
    "SOC Code", "SOC Title", "SOC Match Probability",
    "Typical Education Needed For Entry", "Role Description",
]
detail_cols = [c for c in detail_cols if c in merged.columns]  # tolerate missing education column gracefully

for col_idx, col_name in enumerate(detail_cols, start=1):
    ws.cell(row=1, column=col_idx, value=col_name)
style_header_row(ws, 1, len(detail_cols))

detail_df = merged[detail_cols].reset_index(drop=True)
description_col_idx = detail_cols.index("Role Description") + 1 if "Role Description" in detail_cols else None
for row_idx, row in enumerate(detail_df.itertuples(index=False), start=2):
    for col_idx, val in enumerate(row, start=1):
        cell = ws.cell(row=row_idx, column=col_idx, value=sanitize_for_excel(val))
        cell.font = BODY_FONT
        if col_idx == description_col_idx:
            cell.alignment = WRAP_FONT_ALIGN

ws.freeze_panes = "A2"
col_widths = [22, 32, 26, 12, 12, 32, 14, 10, 32, 12, 26, 60][: len(detail_cols)]
autofit_columns(ws, col_widths)
n_detail_rows = len(detail_df)
print(f"Raw Detail: {n_detail_rows} rows written")

# Column letters for formula references
soc_title_col = get_column_letter(detail_cols.index("SOC Title") + 1)
naics_title_col = get_column_letter(detail_cols.index("NAICS Title") + 1)
hub_col = get_column_letter(detail_cols.index("Program Hub Category") + 1)
last_detail_row = n_detail_rows + 1


def build_summary_sheet(sheet_name, title_col_name_in_detail, title_col_letter):
    ws = wb.create_sheet(sheet_name)
    unique_titles = (
        merged[title_col_name_in_detail].dropna().value_counts().index.tolist()
    )

    headers = [title_col_name_in_detail, "Total Count"] + hubs
    for col_idx, h in enumerate(headers, start=1):
        ws.cell(row=1, column=col_idx, value=h)
    style_header_row(ws, 1, len(headers))

    for row_idx, title in enumerate(unique_titles, start=2):
        ws.cell(row=row_idx, column=1, value=sanitize_for_excel(title)).font = BODY_FONT
        ws.cell(row=row_idx, column=2,
                value=f"=COUNTIF('Raw Detail'!${title_col_letter}$2:${title_col_letter}${last_detail_row},$A{row_idx})"
                ).font = BODY_FONT
        for hub_idx, hub in enumerate(hubs, start=3):
            hub_col_letter = get_column_letter(hub_idx)
            formula = (
                f"=COUNTIFS('Raw Detail'!${title_col_letter}$2:${title_col_letter}${last_detail_row},$A{row_idx},"
                f"'Raw Detail'!${hub_col}$2:${hub_col}${last_detail_row},{hub_col_letter}$1)"
            )
            ws.cell(row=row_idx, column=hub_idx, value=formula).font = BODY_FONT

    ws.freeze_panes = "A2"
    autofit_columns(ws, [45, 14] + [22] * len(hubs))
    print(f"{sheet_name}: {len(unique_titles)} unique titles, formulas referencing Raw Detail")
    return ws


build_summary_sheet("SOC Summary", "SOC Title", soc_title_col)
build_summary_sheet("NAICS Summary", "NAICS Title", naics_title_col)

wb.save(OUT_PATH)
print(f"\nWrote {OUT_PATH}")
print("Run recalc.py on this file next so the COUNTIFS formulas show cached values, "
      "not just formula text, when opened.")

Core hub rows (successfully coded): 6816
Cultural Corps rows (successfully coded): 1149
Combined total: 7965
Hubs present in this data: ['Arts, Entertainment, and Recreation', 'Community and Social Services', 'Healthcare', 'Marketing and Communications', 'STEM and Green']
Raw Detail: 7965 rows written
SOC Summary: 259 unique titles, formulas referencing Raw Detail
NAICS Summary: 121 unique titles, formulas referencing Raw Detail

Wrote ../../data/Sector_Detail_Workbook.xlsx
Run recalc.py on this file next so the COUNTIFS formulas show cached values, not just formula text, when opened.
